In [ ]:
from AlgorithmImports import *
from QuantConnect.Research import QuantBook
from QuantConnect.Data.UniverseSelection import FutureUniverse
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



DATA MAPPING MODE

LAST_TRADING_DAY: DataMappingMode
The contract maps on the previous day of expiration of the front month (0)

FIRST_DAY_MONTH: DataMappingMode
The contract maps on the first date of the delivery month of the front month. If the contract expires prior to this date, then it rolls on the contract's last trading date instead (1)

OPEN_INTEREST: DataMappingMode
The contract maps when the following back month contract has a higher open interest that the current front month (2)

OPEN_INTEREST_ANNUAL: DataMappingMode
The contract maps when any of the back month contracts of the next year have a higher volume that the current front month (3)

DATA NORMALIZATION

FORWARD_PANAMA_CANAL: DataNormalizationMode
Eliminates price jumps between two consecutive contracts, adding a factor based on the difference of their prices. The first contract has the true price. Factor 0. (4)

BACKWARDS_PANAMA_CANAL: DataNormalizationMode
Eliminates price jumps between two consecutive contracts, adding a factor based on the difference of their prices. The last contract has the true price. Factor 0. (5)

BACKWARDS_RATIO: DataNormalizationMode
Eliminates price jumps between two consecutive contracts, multiplying the prices by their ratio. The last contract has the true price. Factor 1. (6)

In [ ]:
start_date = datetime(2014, 1, 1)
exit_date = datetime(2026, 1, 1)

In [ ]:
qb = QuantBook()

futures_list = {
    "gold": Futures.Metals.GOLD,
    "copper": Futures.Metals.COPPER,
    "corn": Futures.Grains.CORN,
    "gasoline": Futures.Energy.GASOLINE,
    "sp500": Futures.Indices.SP_500_E_MINI,
    "treasury_10y": Futures.Financials.Y_10_TREASURY_NOTE,
    "euro": Futures.Currencies.EUR,
}

futures_data = {}

for name, future_symbol in futures_list.items():
    print(name)
    future = qb.add_future(future_symbol)
    future.set_filter(0, 90)

    # Price / OHLC data for each individual contract
    chain_history = qb.future_history(
        future.symbol,
        start=start_date,
        end=exit_date,
        resolution=Resolution.DAILY
    )

    contracts = chain_history.data_frame.reset_index()

    # Open interest for roll selection
    open_interest = qb.history(
        FutureUniverse,
        future.symbol,
        start=start_date,
        end=exit_date,
        flatten=True
    ).reset_index()

    contracts["date"] = contracts["time"].dt.normalize()
    open_interest["date"] = open_interest["time"].dt.normalize()

    contracts = contracts.merge(
        open_interest[
            ["date", "symbol", "openinterest"]
        ],
        on=["date", "symbol"],
        how="left"
    )

    contracts = contracts.drop(columns="date")

    futures_data[name] = contracts

In [ ]:
qb = QuantBook()

future = qb.add_future(
    Futures.Grains.CORN
)

future.set_filter(0, 90)# contracts that expire in 90 days

chain_history = qb.future_history(
    future.symbol,
    start=start_date,
    end=exit_date,
    resolution=Resolution.DAILY
)

contracts = chain_history.data_frame
contracts = contracts.reset_index()

open_interest = qb.history(
    FutureUniverse,
    future.symbol,
    start=start_date,
    end=exit_date,
    flatten=True
).reset_index()

contracts["date"] = contracts["time"].dt.normalize()
open_interest["date"] = open_interest["time"].dt.normalize()

# Not everyday has open interests
contracts = contracts.merge(
    open_interest[["date", "symbol", "openinterest"]],
    on=["date", "symbol"],
    how="left"
)

contracts = contracts.drop(columns="date")

print(contracts.to_string())

In [ ]:
def rollover_org_kindof(current_date, current_contract):
    current_date = pd.Timestamp(current_date).normalize()
    current_contracts = contracts[(contracts["time"].dt.normalize() == current_date) & (contracts["expiry"] > current_contract.expiry)].sort_values(by="expiry")
    ratio = 1.0

    if current_contracts.empty:
        # non trading day
        return (current_contract, ratio)

    new_contract = contracts[(contracts["time"].dt.normalize() == current_date) & (contracts["symbol"] == current_contract.symbol)].iloc[0]

    # Quant Connect uses same day open interest (not possible as it is posted at the end of day)
    back_month_contract = current_contracts.iloc[0]
    if current_date == current_contract.expiry or (pd.notna(new_contract.openinterest) and back_month_contract.openinterest >= new_contract.openinterest):
        new_contract= current_contracts[current_contracts["symbol"] == back_month_contract.symbol].iloc[0]


    # Quant Connect looks ahead for open interest if current day open interest is not reported (sometimes jumps by more than one month on scare occasions (not implemented here))
    if pd.isna(new_contract.openinterest):
        next_date=current_date+ pd.Timedelta(days=1)
        next_contracts = contracts[(contracts["time"].dt.normalize() == next_date) & (contracts["expiry"] > current_contract.expiry)].sort_values(by="expiry")

        if next_contracts.empty:
            # If no immediate next day keep current contract
            return (new_contract, ratio)

        next_current_contract = contracts[(contracts["time"].dt.normalize() == next_date) & (contracts["symbol"] == current_contract.symbol)].iloc[0]
        next_back_month_contract = next_contracts.iloc[0]
        if next_back_month_contract.openinterest >= next_current_contract.openinterest:
            new_contract= current_contracts[current_contracts["symbol"] == back_month_contract.symbol].iloc[0]

    if new_contract.symbol != current_contract.symbol:
        old_contracts = contracts[contracts["time"].dt.normalize() == current_contract.time.normalize()]
        old_contract_of_new_contract = old_contracts[old_contracts["symbol"] == new_contract.symbol].iloc[0]
        ratio =  old_contract_of_new_contract.close / current_contract.close



    return (new_contract, ratio)

In [ ]:
def rollover(current_date, current_contract, contracts):
    current_date = pd.Timestamp(current_date).normalize()
    current_contracts = contracts[contracts["time"].dt.normalize() == current_date]
    ratio = 1.0

    if current_contracts.empty:
        # non trading day
        return (current_contract, ratio)

    new_contract = current_contracts[current_contracts["symbol"] == current_contract.symbol].iloc[0]

    previous_dates = contracts.loc[contracts["time"] < current_date]

    if previous_dates.empty:
        # first trading day
        return (new_contract, ratio)
    
    # Use last trading days data to decide rollover
    prev_date = previous_dates["time"].max().normalize()

    prev_contracts = contracts[(contracts["time"].dt.normalize() == prev_date) & (contracts["expiry"] > current_contract.expiry)].sort_values(by="expiry")

    if prev_contracts.empty:
        latest_contract = contracts.loc[contracts["expiry"].idxmax()]
        if current_contract.expiry != latest_contract.expiry:
            print("should not happen")
        return (new_contract, ratio)
    
    # OPEN_INTEREST on just next month
    back_month_contract = prev_contracts.iloc[0]
    if current_date.normalize() == current_contract.expiry.normalize() or (pd.notna(current_contract.openinterest) and back_month_contract.openinterest >= current_contract.openinterest):
        new_contract= current_contracts[current_contracts["symbol"] == back_month_contract.symbol].iloc[0]

    if new_contract.symbol != current_contract.symbol:
        ratio =  back_month_contract.close / current_contract.close

    return (new_contract, ratio)

In [ ]:
ROLLOVER_TYPE = 0
NORMALIZATION = True

In [ ]:
def extract (start_date, exit_date, contracts):
    entry_date = pd.Timestamp(start_date).normalize()
    exit_date = pd.Timestamp(exit_date).normalize()

    # Pick entry contract
    while True:
        entry_contracts = contracts[contracts["time"].dt.normalize() == entry_date]
        if not entry_contracts.empty:
            break
        print("no data on day: ", entry_date, "picking next day")
        entry_date += pd.Timedelta(days=1)

    # Choose earliest max volume contract
    max_volume = entry_contracts["volume"].max()
    highest_volume_contracts = entry_contracts[entry_contracts["volume"] == max_volume]
    highest_volume_contract = highest_volume_contracts.iloc[0]
    
    print("Picking contract with symbol: ", highest_volume_contract.symbol, "expiry: ", highest_volume_contract.expiry)

    # data on each day
    rows=[]
    ratios = []
    current_contract = highest_volume_contract
    for current_date in pd.date_range(entry_date, exit_date, inclusive="left"):
        if ROLLOVER_TYPE ==0:
            (current_contract, ratio) = rollover(current_date, current_contract, contracts)
        else:
            # deprecated
            (current_contract, ratio) = rollover_org_kindof(current_date, current_contract)
        
        if not(np.isclose(ratio, 1.0)):
            ratios.append((current_date,ratio))

        row = current_contract.to_dict()  # copies every column
        row["current_date"] = current_date
        rows.append(row)

    df = pd.DataFrame(rows)

    if NORMALIZATION:
        price_columns = [
            "open", "high", "low", "close",
            "askopen", "askhigh", "asklow", "askclose",
            "bidopen", "bidhigh", "bidlow", "bidclose"
        ]


        for (time, ratio) in ratios:
            mask = df["time"].dt.normalize() < time
            df.loc[mask, price_columns] *= ratio

    return (entry_date, df)

In [ ]:
price_series = {}

entry_date = start_date

for k, contracts in futures_data.items():
    (e_date, data) = extract(start_date, exit_date, contracts)
    price_series[k] = data

    entry_date = max(e_date, entry_date)

print(entry_date)


In [ ]:
def compare(impl, org):
    columns_to_compare = [
        "askclose", "askhigh", "asklow", "askopen", "asksize",
        "bidclose", "bidhigh", "bidlow", "bidopen", "bidsize",
        "close", "high", "low", "open", "volume"
    ]

    impl = impl.copy()
    org = org.copy()

    # Restore index columns if necessary
    if "current_date" not in impl.columns and "time" not in impl.columns:
        impl = impl.reset_index()

    if "time" not in org.columns:
        org = org.reset_index()

    # impl uses current_date; org uses time
    if "current_date" in impl.columns:
        impl_date_column = "current_date"
    elif "time" in impl.columns:
        impl_date_column = "time"
    else:
        raise KeyError("impl must contain either 'current_date' or 'time'")

    if "time" not in org.columns:
        raise KeyError("org must contain a 'time' column")

    # Compare calendar dates rather than exact timestamps
    impl["_compare_date"] = pd.to_datetime(
        impl[impl_date_column]
    ).dt.normalize()

    org["_compare_date"] = pd.to_datetime(
        org["time"]
    ).dt.normalize()

    # Unique ID prevents duplicate org rows from duplicating final output
    impl["_impl_row_id"] = np.arange(len(impl))

    merged = impl.merge(
        org,
        on="_compare_date",
        how="left",
        suffixes=("_impl", "_org"),
        indicator=True
    )

    org_row_exists = merged["_merge"].eq("both")

    values_match = np.logical_and.reduce([
        np.isclose(
            merged[f"{column}_impl"],
            merged[f"{column}_org"],
            equal_nan=True
        )
        for column in columns_to_compare
    ])

    # A candidate matches only when org actually has that row
    merged["_candidate_match"] = org_row_exists & values_match

    # Reduce multiple org rows back to one result per impl row
    result = (
        merged.groupby("_impl_row_id", sort=False)
        .agg(
            org_exists=("_merge", lambda values: values.eq("both").any()),
            values_match=("_candidate_match", "any")
        )
        .reset_index()
    )

    # Missing from org counts as True
    result["same"] = (
        ~result["org_exists"]
        | result["values_match"]
    )

    # Restore the original impl date/time for display
    result = result.merge(
        impl[["_impl_row_id", impl_date_column]],
        on="_impl_row_id",
        how="left"
    )

    result = result[
        [impl_date_column, "org_exists", "same"]
    ].rename(columns={impl_date_column: "time"})

    print(result.to_string(index=False))
    print("\nAll true:", result["same"].all())



In [ ]:
# Quant Connect decides rollovers based on same day open interest, and also when open interest is NaN, it probably uses next days open interest to determine a skip
# USE ROLLOVER_TYPE=1 AND NORMALIZATION = 1 AND SET data_normalization_mode=DataNormalizationMode.RAW
compare(data, history_org)

In [ ]:
# ─── Rollover unit test helpers ───────────────────────────────────────────────
_saved_contracts = contracts.copy()
_results = []

def _check(name, cond, detail=""):
    if cond:
        _results.append(("PASS", name))
        print(f"  \033[32mPASS\033[0m  {name}")
    else:
        _results.append(("FAIL", name))
        extra = f"\n         {detail}" if detail else ""
        print(f"  \033[31mFAIL\033[0m  {name}{extra}")

def _row(sym, expiry, time, close, oi=np.nan, **kw):
    return dict(
        symbol=sym, expiry=pd.Timestamp(expiry), time=pd.Timestamp(time),
        close=float(close), open=float(close), high=float(close),
        low=float(close), volume=1000.0, openinterest=oi, **kw,
    )

_CLF, _CLG = "CLF24", "CLG24"
_EF  = "2024-01-31"
_EG  = "2024-02-29"
_D0  = pd.Timestamp("2024-01-10")   # prev trading day
_D1  = pd.Timestamp("2024-01-11")   # current trading day

print("Test helpers loaded.")

In [ ]:
# ─── Tests 1-6: rollover decision logic ──────────────────────────────────────

# 1 · non-trading day → same contract passed back, ratio = 1
print("1 · non-trading day → same contract, ratio=1")
contracts = pd.DataFrame([_row(_CLF, _EF, _D0, 80.0, 1000)])
stub = contracts.iloc[0]
c, r = rollover(_D1, stub)   # _D1 has no rows in contracts
_check("same symbol returned",  c.symbol == _CLF)
_check("ratio = 1.0",           np.isclose(r, 1.0))

# 2 · first trading day (nothing before it) → today's row, ratio = 1
print("\n2 · first trading day (no previous rows) → today's row, ratio=1")
contracts = pd.DataFrame([_row(_CLF, _EF, _D1, 80.0, 1000)])
stub = contracts.iloc[0]
c, r = rollover(_D1, stub)
_check("today's row returned",  c.symbol == _CLF)
_check("ratio = 1.0",           np.isclose(r, 1.0))

# 3 · front OI > back OI → no rollover
print("\n3 · front OI 1000 > back OI 800 → no rollover")
contracts = pd.DataFrame([
    _row(_CLF, _EF, _D0, 80.0, 1000),
    _row(_CLG, _EG, _D0, 80.5,  800),
    _row(_CLF, _EF, _D1, 81.0),
    _row(_CLG, _EG, _D1, 81.5),
])
prev_front = contracts[(contracts.time == _D0) & (contracts.symbol == _CLF)].iloc[0]
c, r = rollover(_D1, prev_front)
_check("stays on front contract",  c.symbol == _CLF)
_check("ratio = 1.0",              np.isclose(r, 1.0))

# 4 · back OI ≥ front OI → rollover, ratio uses PREVIOUS-day closes
print("\n4 · back OI 800 ≥ front OI 600 → rollover")
contracts = pd.DataFrame([
    _row(_CLF, _EF, _D0, 80.0,  600),
    _row(_CLG, _EG, _D0, 80.5,  800),
    _row(_CLF, _EF, _D1, 81.0),
    _row(_CLG, _EG, _D1, 81.5),
])
prev_front = contracts[(contracts.time == _D0) & (contracts.symbol == _CLF)].iloc[0]
c, r = rollover(_D1, prev_front)
exp_ratio = 80.5 / 80.0
_check("switches to back month",          c.symbol == _CLG)
_check("ratio = back_D0_close / front_D0_close",
       np.isclose(r, exp_ratio),
       f"got {r:.6f}, expected {exp_ratio:.6f}")

# 5 · expiry forces rollover regardless of OI dominance
print("\n5 · front expires today → always rolls even when front OI > back OI")
contracts = pd.DataFrame([
    _row(_CLF, _D1,  _D0, 80.0, 1000),  # front expires on _D1, OI dominant
    _row(_CLG, _EG,  _D0, 80.5,  300),  # back has lower OI
    _row(_CLF, _D1,  _D1, 81.0),
    _row(_CLG, _EG,  _D1, 81.5),
])
prev_front = contracts[(contracts.time == _D0) & (contracts.symbol == _CLF)].iloc[0]
c, r = rollover(_D1, prev_front)
_check("rolls to back on expiry",  c.symbol == _CLG)
_check("ratio ≠ 1.0",             not np.isclose(r, 1.0))

# 6 · NaN front OI → pd.notna check fails → OI condition skipped → no rollover
print("\n6 · front OI = NaN → OI condition skipped → no rollover")
contracts = pd.DataFrame([
    _row(_CLF, _EF, _D0, 80.0, np.nan),  # front OI unknown
    _row(_CLG, _EG, _D0, 80.5,  800),
    _row(_CLF, _EF, _D1, 81.0),
    _row(_CLG, _EG, _D1, 81.5),
])
prev_front = contracts[(contracts.time == _D0) & (contracts.symbol == _CLF)].iloc[0]
c, r = rollover(_D1, prev_front)
_check("stays on front (NaN OI skips check)",  c.symbol == _CLF)
_check("ratio = 1.0",                          np.isclose(r, 1.0))

In [ ]:
# ─── Test 7: ratio eliminates artificial price jump at rollover ───────────────
#
# Property: front_D0_close * ratio == back_D0_close
# This means adjusting historical prices by the ratio makes the series
# continuous — the last pre-rollover adjusted close equals the back contract's
# close on that same day, so no fake return appears in the price series.
#
# Timeline:
#   D_prev → D0: front OI still dominant (no rollover yet)
#   D0    → D1: back OI crosses front OI  → rollover on D1

print("7 · price continuity: front_D0_close × ratio == back_D0_close")
_D_prev = pd.Timestamp("2024-01-09")
contracts = pd.DataFrame([
    _row(_CLF, _EF, _D_prev, 79.0, 1200),
    _row(_CLG, _EG, _D_prev, 79.5,  900),
    _row(_CLF, _EF, _D0,     80.0,  600),   # OI crosses on D0
    _row(_CLG, _EG, _D0,     80.5,  800),
    _row(_CLF, _EF, _D1,     81.0),
    _row(_CLG, _EG, _D1,     81.5),
])

clf_d_prev = contracts[(contracts.time == _D_prev) & (contracts.symbol == _CLF)].iloc[0]

# D0 iteration: back OI 900 < front OI 1200 → no rollover
c0, r0 = rollover(_D0, clf_d_prev)
_check("D0: stays on CLF  (back OI 900 < front OI 1200)",  c0.symbol == _CLF)
_check("D0: ratio = 1.0",                                  np.isclose(r0, 1.0))

# D1 iteration: back OI 800 ≥ front OI 600 → rollover
c1, r1 = rollover(_D1, c0)
_check("D1: rolls to CLG  (back OI 800 ≥ front OI 600)",  c1.symbol == _CLG)

clg_d0 = contracts[(contracts.time == _D0) & (contracts.symbol == _CLG)].iloc[0]
adj_clf_d0_close = c0.close * r1   # what D0's close becomes after backward-ratio adjustment

_check("adj_D0_close == back_D0_close  (series joins cleanly at rollover)",
       np.isclose(adj_clf_d0_close, clg_d0.close),
       f"CLF_D0={c0.close} × ratio={r1:.5f} = {adj_clf_d0_close:.4f},  CLG_D0={clg_d0.close}")

# Corollary: close-to-close return through the rollover equals actual CLG return
clg_d1 = contracts[(contracts.time == _D1) & (contracts.symbol == _CLG)].iloc[0]
ret_actual = (clg_d1.close - clg_d0.close) / clg_d0.close
ret_series = (clg_d1.close - adj_clf_d0_close) / adj_clf_d0_close
_check("return through rollover == actual CLG return  (no fake price move)",
       np.isclose(ret_actual, ret_series),
       f"actual={ret_actual:.6f},  series={ret_series:.6f}")

# ─── Restore global contracts and print summary ───────────────────────────────
contracts = _saved_contracts

passed = sum(1 for s, _ in _results if s == "PASS")
failed = sum(1 for s, _ in _results if s == "FAIL")
print(f"\n{'─'*50}")
print(f"  {passed} passed · {failed} failed")
if failed:
    print("\n  Failed tests:")
    for s, name in _results:
        if s == "FAIL":
            print(f"    ✗  {name}")

In [ ]:
# ─── Tests 8-11: edge cases ───────────────────────────────────────────────────
_CLH = "CLH24"
_EH  = "2024-03-28"

# 8 · OI exactly equal (back == front) → should still roll  (condition is >=, not >)
print("8 · back OI == front OI → rolls  (>= not just >)")
contracts = pd.DataFrame([
    _row(_CLF, _EF, _D0, 80.0, 1000),
    _row(_CLG, _EG, _D0, 80.5, 1000),   # exactly tied
    _row(_CLF, _EF, _D1, 81.0),
    _row(_CLG, _EG, _D1, 81.5),
])
prev_front = contracts[(contracts.time == _D0) & (contracts.symbol == _CLF)].iloc[0]
c, r = rollover(_D1, prev_front)
_check("rolls when OI tied",             c.symbol == _CLG)
_check("ratio = back_D0 / front_D0",     np.isclose(r, 80.5 / 80.0))

# 9 · three contracts available → picks nearest-expiry back month, ignores further out
# (back month candidate is sorted by expiry ascending and .iloc[0] is taken)
print("\n9 · three contracts → rolls to nearest expiry back month, not furthest")
contracts = pd.DataFrame([
    _row(_CLF, _EF, _D0, 80.0,  500),
    _row(_CLG, _EG, _D0, 80.5,  800),   # nearest back month — should win
    _row(_CLH, _EH, _D0, 81.0,  900),   # further back month, even higher OI — must NOT win
    _row(_CLF, _EF, _D1, 81.0),
    _row(_CLG, _EG, _D1, 81.5),
    _row(_CLH, _EH, _D1, 82.0),
])
prev_front = contracts[(contracts.time == _D0) & (contracts.symbol == _CLF)].iloc[0]
c, r = rollover(_D1, prev_front)
_check("picks CLG not CLH  (nearest expiry wins)",  c.symbol == _CLG)
_check("ratio uses CLG_D0 / CLF_D0",               np.isclose(r, 80.5 / 80.0))

# 10 · non-consecutive trading days (weekend / holiday gap)
# prev_date is found via .max() so a 3-day gap should still resolve correctly
print("\n10 · weekend gap: prev trading day is 3 calendar days back")
_FRIDAY = pd.Timestamp("2024-01-05")
_MONDAY = pd.Timestamp("2024-01-08")
contracts = pd.DataFrame([
    _row(_CLF, _EF, _FRIDAY, 80.0,  600),   # last trading day before weekend
    _row(_CLG, _EG, _FRIDAY, 80.5,  800),
    # Saturday, Sunday intentionally absent
    _row(_CLF, _EF, _MONDAY, 81.0),
    _row(_CLG, _EG, _MONDAY, 81.5),
])
prev_front = contracts[(contracts.time == _FRIDAY) & (contracts.symbol == _CLF)].iloc[0]
c, r = rollover(_MONDAY, prev_front)
_check("rolls to CLG across weekend gap",        c.symbol == _CLG)
_check("ratio uses Friday closes (not Monday)",  np.isclose(r, 80.5 / 80.0))

# 11 · current contract is already the back month (CLG); next candidate is CLH
# verifies that "back month" is always relative to whatever contract we're on
print("\n11 · already on CLG → next back month is CLH")
contracts = pd.DataFrame([
    _row(_CLG, _EG, _D0, 80.0,  400),
    _row(_CLH, _EH, _D0, 80.5,  700),   # CLH OI dominant → should roll CLG→CLH
    _row(_CLG, _EG, _D1, 81.0),
    _row(_CLH, _EH, _D1, 81.5),
])
prev_clg = contracts[(contracts.time == _D0) & (contracts.symbol == _CLG)].iloc[0]
c, r = rollover(_D1, prev_clg)
_check("rolls CLG → CLH when CLH OI dominant",  c.symbol == _CLH)
_check("ratio = CLH_D0 / CLG_D0",               np.isclose(r, 80.5 / 80.0))

In [ ]:
def calculate_volatility(data, lda):
    trading = data[data["daily_ret"] != 0].copy()

    first_60_data = trading.iloc[:60].copy()
    sigma_0 = ((first_60_data["daily_ret"] ** 2).mean()) ** 0.5

    trading["rolling_std"] = 0.0

    index_std = trading.columns.get_loc("rolling_std")
    index_rt = trading.columns.get_loc("daily_ret")

    trading.iloc[59, index_std] = sigma_0
    for day in range(60, len(trading)):
        trading.iloc[day, index_std] = (trading.iloc[day-1, index_std] ** 2 * lda + (1-lda) * trading.iloc[day-1, index_rt] ** 2) ** 0.5

    data["rolling_std"] = trading["rolling_std"]
    data["rolling_std"] = data["rolling_std"].ffill().fillna(0.0)


def simulate (data, enter_date, exit_date, lookbacks, AUM_0, target_vol):
    investment_amo = 0
    for current_date in pd.date_range(enter_date, exit_date, inclusive="left"): 
        curr_day = data[data["current_date"] == current_date].iloc[0]
        prev_day = data[data["current_date"] < current_date].iloc[-1]  # last trading day, not calendar day-1

        if current_date.month != prev_day.current_date.month or current_date.year != prev_day.current_date.year: # new month
            overall_trend = 0
            for lookback in lookbacks:
                lookback_date = prev_day.current_date - pd.DateOffset(months = lookback)
                lookback_close = data[data["current_date"] <= lookback_date].iloc[-1].close  # nearest trading day on or before
                price_move = prev_day.close - lookback_close

                trend = 1
                if price_move < 0:
                    trend = -1
                overall_trend +=trend

            annualized_std = prev_day.rolling_std * (252 ** 0.5)
            if annualized_std == 0:
                investment_amo = 0
            else:
                investment_amo = prev_day.strat_amo * (overall_trend / len(lookbacks)) * (target_vol / annualized_std)

        data.loc[data["current_date"] == current_date, "strat_amo"] = prev_day.strat_amo + curr_day.daily_ret * investment_amo
        data.loc[data["current_date"] == current_date, "hold_amo"] = prev_day.hold_amo * (1+curr_day.daily_ret)


def model (data, enter_date, exit_date, lda, lookback, aum_0, target_vol):
    data["daily_ret"] = (data["close"] - data["close"].shift(1)) / data["close"].shift(1)
    data["strat_amo"] = aum_0
    data["hold_amo"] = aum_0

    calculate_volatility(data, lda)

    simulate(data, enter_date, exit_date, lookback, aum_0, target_vol)



In [ ]:
LOOKBACKS = [1,3 ,12]
AUM_0 = 100000.0
LBDA = 0.94
TARGET_VOL = 0.1

ENTER_DATE = entry_date + pd.DateOffset(months=max(LOOKBACKS))
EXIT_DATE = exit_date

In [ ]:
data["daily_ret"] = (data["close"] - data["close"].shift(1)) / data["close"].shift(1)
data["strat_amo"] = AUM_0

calculate_volatility(data, lda)

simulate(enter_date, exit_date, lookbacks[3], AUM_0, target_vol, data)

In [ ]:
def graph(data, lookbacks):
    plot_data = data[
        pd.to_datetime(data["current_date"]) >= ENTER_DATE
    ].copy()

    lookback_str = "_".join(map(str, lookbacks))

    plt.figure(figsize=(12, 6))

    plt.plot(
        plot_data["current_date"],
        plot_data["strat_amo"],
        label="Strategy"
    )

    plt.plot(
        plot_data["current_date"],
        plot_data["hold_amo"],
        label="Buy and Hold"
    )

    plt.xlabel("Date")
    plt.ylabel("Portfolio Value")
    plt.title(f"Strategy vs Buy and Hold — Lookbacks: {lookbacks}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    plt.savefig(f"lookbacks_{lookback_str}.png")
    # plt.show()

    # plt.close()


def print_metrics(data, lookbacks, value_col="strat_amo"):
    values = data[value_col].dropna()
    returns = values.pct_change().dropna()

    total_return = values.iloc[-1] / values.iloc[0] - 1

    annual_vol = returns.std() * np.sqrt(252)

    sharpe = (
        returns.mean() / returns.std() * np.sqrt(252)
        if returns.std() != 0
        else np.nan
    )

    drawdown = values / values.cummax() - 1
    max_drawdown = drawdown.min()

    print(f"\nLookbacks: {lookbacks}")
    print(f"Final value:    ${values.iloc[-1]:,.2f}")
    print(f"Total return:   {total_return:.2%}")
    print(f"Annual vol:     {annual_vol:.2%}")
    print(f"Sharpe ratio:   {sharpe:.3f}")
    print(f"Max drawdown:   {max_drawdown:.2%}")

In [ ]:
portfolio_returns = []


for k, data in price_series.items():
    print("\n", k)
    run_data = data.copy()
    model(run_data, ENTER_DATE, EXIT_DATE, LBDA, LOOKBACKS, AUM_0, TARGET_VOL)

    print_metrics(run_data, LOOKBACKS)

    temp = run_data[
        ["current_date", "strat_amo", "hold_amo"]
    ].copy()

    temp[f"{k}_strat"] = temp["strat_amo"].pct_change()
    temp[f"{k}_hold"] = temp["hold_amo"].pct_change()

    temp = temp[
        ["current_date", f"{k}_strat", f"{k}_hold"]
    ]

    portfolio_returns.append(temp)
    

In [ ]:
PORTFOLIO_TARGET_VOL = 0.10
PORTFOLIO_VOL_LOOKBACK = 60
MAX_PORTFOLIO_LEVERAGE = 3.0

In [ ]:
# ==========================================
# Merge all futures
# ==========================================

portfolio = portfolio_returns[0].copy()

for df in portfolio_returns[1:]:
    portfolio = portfolio.merge(
        df,
        on="current_date",
        how="inner"
    )

portfolio = portfolio.sort_values(
    "current_date"
).reset_index(drop=True)


# ==========================================
# Separate strategy and buy-and-hold columns
# ==========================================

strat_cols = [
    col for col in portfolio.columns
    if col.endswith("_strat")
]

hold_cols = [
    col for col in portfolio.columns
    if col.endswith("_hold")
]


# ==========================================
# Remove days when ALL markets are closed
# ==========================================

portfolio = portfolio[
    portfolio[hold_cols].abs().sum(axis=1) > 0
].copy()

portfolio = portfolio.reset_index(drop=True)


# ==========================================
# Equal-weight RAW portfolio returns
# ==========================================

portfolio["raw_strat_return"] = (
    portfolio[strat_cols].mean(axis=1)
)

portfolio["hold_return"] = (
    portfolio[hold_cols].mean(axis=1)
)


# ==========================================
# Estimate portfolio volatility
# Using ONLY information available before today
# ==========================================

portfolio["portfolio_vol"] = (
    portfolio["raw_strat_return"]
    .rolling(PORTFOLIO_VOL_LOOKBACK)
    .std()
    .shift(1)
    * np.sqrt(252)
)


# ==========================================
# Portfolio volatility scaling
# ==========================================

portfolio["portfolio_leverage"] = (
    PORTFOLIO_TARGET_VOL
    / portfolio["portfolio_vol"]
)

portfolio["portfolio_leverage"] = (
    portfolio["portfolio_leverage"]
    .replace([np.inf, -np.inf], np.nan)
    .clip(upper=MAX_PORTFOLIO_LEVERAGE)
)


# ==========================================
# Need historical portfolio vol first
# Drop warmup period for BOTH strategies
# ==========================================

portfolio = portfolio.dropna(
    subset=["portfolio_leverage"]
).copy()

portfolio = portfolio.reset_index(drop=True)


# ==========================================
# Apply today's leverage to today's return
# ==========================================

portfolio["strat_return"] = (
    portfolio["raw_strat_return"]
    * portfolio["portfolio_leverage"]
)


# ==========================================
# Build portfolio equity curves
# ==========================================

portfolio["strat_amo"] = (
    AUM_0 *
    (1 + portfolio["strat_return"]).cumprod()
)

portfolio["hold_amo"] = (
    AUM_0 *
    (1 + portfolio["hold_return"]).cumprod()
)


# ==========================================
# Metrics
# ==========================================

def portfolio_metrics(values, returns, name):
    values = values.dropna()
    returns = returns.dropna()

    total_return = values.iloc[-1] / AUM_0 - 1

    annual_vol = (
        returns.std() * np.sqrt(252)
    )

    sharpe = (
        returns.mean()
        / returns.std()
        * np.sqrt(252)
        if returns.std() != 0
        else np.nan
    )

    drawdown = (
        values / values.cummax() - 1
    )

    max_drawdown = drawdown.min()

    print(f"\n===== {name} =====")
    print(f"Final value:     ${values.iloc[-1]:,.2f}")
    print(f"Total return:    {total_return:.2%}")
    print(f"Annual vol:      {annual_vol:.2%}")
    print(f"Sharpe ratio:    {sharpe:.3f}")
    print(f"Max drawdown:    {max_drawdown:.2%}")


portfolio_metrics(
    portfolio["strat_amo"],
    portfolio["strat_return"],
    "VOL TARGETED TREND STRATEGY"
)

portfolio_metrics(
    portfolio["hold_amo"],
    portfolio["hold_return"],
    "BUY AND HOLD"
)


# ==========================================
# Extra diagnostics
# ==========================================

print(
    "\nAverage portfolio leverage:",
    portfolio["portfolio_leverage"].mean()
)

print(
    "Median portfolio leverage:",
    portfolio["portfolio_leverage"].median()
)

print(
    "Max portfolio leverage:",
    portfolio["portfolio_leverage"].max()
)

print(
    "Average estimated portfolio vol:",
    portfolio["portfolio_vol"].mean()
)


# ==========================================
# Graph
# ==========================================

plt.figure(figsize=(12, 6))

plt.plot(
    portfolio["current_date"],
    portfolio["strat_amo"],
    label="Vol Targeted Trend Strategy"
)

plt.plot(
    portfolio["current_date"],
    portfolio["hold_amo"],
    label="Buy and Hold"
)

plt.xlabel("Date")
plt.ylabel("Portfolio Value")

plt.title(
    f"Trend Strategy vs Buy and Hold — Lookbacks: {LOOKBACKS}"
)

plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig("total_portfolio.png")
plt.show()